In [13]:
import os
os.chdir(r"D:\Study\Programs\trading")

In [14]:
from strategies_dev.utils import resample_fractional_minute
from pathlib import Path
import pandas as pd
import numpy as np

In [25]:
date_ = "27APR2026"
file_name = "NIFTY26APR23950PE.xlsx"
N_list = [1, 4, 5, 6, 7]
file_path = Path(fr"D:\Study\Programs\trading\assets\logs\{date_}\extracted_symbols\{file_name}")
df = pd.read_excel(file_path)

In [26]:
if "PE" in file_name or "CE" in file_name:
    col_name = "last_trade_time"
    df["volume_at_tick"] = df["volume_traded"].diff()
    df.loc[df["volume_at_tick"] == 0, "volume_at_tick"] = np.nan
else:
    col_name = "local_time"

df[col_name] = pd.to_datetime(df[col_name])

In [27]:
for N in N_list:
    clubbed_data = resample_fractional_minute(df, col_name, n=N, depth=False)
    clubbed_data["bucket_time"] = pd.to_datetime(clubbed_data["bucket_time"])
    clubbed_data["date_str"] = clubbed_data["bucket_time"].dt.strftime("%d%b%Y").str.upper()
    clubbed_data = clubbed_data[clubbed_data["date_str"] == date_]
    clubbed_data = clubbed_data.drop(columns=["date_str"])
    fp = Path(fr"D:\Study\Programs\trading\assets\logs\{date_}\candles\{file_name.split(".")[0]}_N={N}.xlsx")
    clubbed_data.to_excel(fp, index=False)